In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_movies")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_movies (
  movie_id BIGINT,
  date_key BIGINT,
  popularity DOUBLE,
  vote_average DOUBLE,
  vote_count INT,
  revenue BIGINT,
  budget BIGINT
)
""")

In [0]:
#controlas tipos explícitamente
#no hay NaN raros
#no hay Int64 de pandas

from pyspark.sql.functions import col, to_date, date_format

df = spark.table("workspace.silver.movies")

fact = (
    df.select(
        col("movie_id").cast("bigint"),
        date_format(to_date(col("release_date")), "yyyyMMdd").cast("bigint").alias("date_key"),
        col("popularity").cast("double"),
        col("vote_average").cast("double"),
        col("vote_count").cast("int"),
        col("revenue").cast("bigint"),
        col("budget").cast("bigint")
    )
    .dropDuplicates(["movie_id"])
)

In [0]:
fact.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_movies")